In [1]:
import pandas as pd  
import numpy as np
from tqdm import tqdm  
from collections import defaultdict  
import os, math, warnings, math, pickle
from tqdm import tqdm
import faiss
import collections
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
warnings.filterwarnings('ignore')


In [2]:
# data_path = './data_raw/'
data_path = './tcdata/' # 天池平台路径
save_path = './temp_results/'  # 天池平台路径

# 定义一个多路召回的字典，将各路召回的结果都保存在这个字典当中
user_multi_recall_dict =  {'itemcf_sim_itemcf_recall': {},
                           'embedding_sim_item_recall': {},
                           'youtubednn_recall': {},
                           'youtubednn_usercf_recall': {}, 
                           'cold_start_recall': {}}
# 是否计算指标
metric_recall = True

In [3]:
from utils.data_utils import get_all_click_df, get_item_topk_click

I0000 00:00:1774425089.905245   17549 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774425089.956164   17549 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774425090.855614   17549 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
# 读取文章的基本属性
from utils.data_utils import get_item_info_df, get_item_emb_dict, get_user_item_time, get_item_user_time_dict

In [5]:
max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))

In [6]:
# 采样数据
# all_click_df = get_all_click_sample(data_path)

# 全量训练集
all_click_df = get_all_click_df(offline=False)

# 对时间戳进行归一化,用于在关联规则的时候计算权重
all_click_df['click_timestamp'] = all_click_df[['click_timestamp']].apply(max_min_scaler)

In [7]:
item_info_df = get_item_info_df(data_path)
# item_emb_dict = get_item_emb_dict(data_path, save_path)

In [8]:
# 获取文章id对应的基本属性，保存成字典的形式，方便后面召回阶段，冷启动阶段直接使用
from utils.data_utils import get_item_info_dict,get_user_hist_item_info_dict,get_hist_and_last_click


In [9]:
# 获取文章的属性信息，保存成字典的形式方便查询
item_type_dict, item_words_dict, item_created_time_dict = get_item_info_dict(item_info_df)

In [10]:
# 提取最后一次点击作为召回评估，如果不需要做召回评估直接使用全量的训练集进行召回(线下验证模型)
# 如果不是召回评估，直接使用全量数据进行召回，不用将最后一次提取出来
trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)

In [11]:
from metrics import metrics_recall
from models.CF import itemcf_sim

In [12]:
# i2i_sim = itemcf_sim(all_click_df, item_created_time_dict)

In [13]:
# from models.CF import usercf_sim
# # 由于usercf计算时候太耗费内存了，这里就不直接运行了
# # 如果是采样的话，是可以运行的
# user_activate_degree_dict = get_user_activate_degree_dict(all_click_df)
# u2u_sim = usercf_sim(all_click_df, user_activate_degree_dict)

In [14]:
# item_emb_df = pd.read_csv(data_path + '/articles_emb.csv')
# emb_i2i_sim = embdding_sim(all_click_df, item_emb_df, save_path, topk=10) # topk可以自行设置

## 向量召回

In [15]:
# from models.VectorSim import embdding_sim
# item_emb_df = pd.read_csv(data_path + '/articles_emb.csv')
# emb_i2i_sim = embdding_sim(all_click_df, item_emb_df, save_path, topk=10) # topk可以自行设置

## 双塔召回

In [17]:
import random
import pickle
import collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder


from models.DNN import TorchYoutubeDNN as YoutubeDNN, YoutubeDNNTrainer, u2u_embdding_sim
from metrics import metrics_recall
from utils.data_utils import gen_data_set, gen_model_input, bucketize_words_count

In [29]:
def youtubednn_u2i_dict(data, topk=20, save_path='./temp_results/', data_path='./tcdata/', words_bucket_num=20):
    SEQ_LEN = 30

    item_info_df = get_item_info_df(data_path)
    data = data.merge(
        item_info_df[['click_article_id', 'category_id', 'words_count']],
        how='left',
        on='click_article_id'
    ).copy()

    data['words_count'] = bucketize_words_count(data['words_count'], bucket_num=words_bucket_num)

    user_profile_ = data[['user_id']].drop_duplicates('user_id')
    item_profile_ = data[['click_article_id']].drop_duplicates('click_article_id')

    features = ['click_article_id', 'user_id', 'category_id', 'words_count']
    feature_max_idx = {}

    data = data.copy()
    for feature in features:
        lbe = LabelEncoder()
        data[feature] = lbe.fit_transform(data[feature]) + 1
        feature_max_idx[feature] = data[feature].max() + 1

    user_profile = data[['user_id']].drop_duplicates('user_id')
    item_profile = data[['click_article_id', 'category_id', 'words_count']].drop_duplicates('click_article_id')

    user_index_2_rawid = dict(zip(user_profile['user_id'], user_profile_['user_id']))
    item_index_2_rawid = dict(zip(item_profile['click_article_id'], item_profile_['click_article_id']))

    item_feat_dict = {
        'category_id': dict(zip(item_profile['click_article_id'], item_profile['category_id'])),
        'words_count': dict(zip(item_profile['click_article_id'], item_profile['words_count'])),
    }

    print("构造数据集")
    train_set, test_set = gen_data_set(data, 0)

    train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN, item_feat_dict=item_feat_dict)
    test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN, item_feat_dict=item_feat_dict)

    embedding_dim = 16
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = YoutubeDNN(
        user_num=feature_max_idx['user_id'],
        item_num=feature_max_idx['click_article_id'],
        embedding_dim=embedding_dim,
        hidden_units=(128, embedding_dim),
        item_hidden_units=(128, embedding_dim),
        padding_idx=0,
        category_num=feature_max_idx['category_id'],
        words_num=feature_max_idx['words_count'],
    ).to(device)

    trainer = YoutubeDNNTrainer(
        model,
        device,
        batch_size=512,
        lr=1e-3,
        epochs=10
    )

    print("开始训练")
    trainer.fit(train_model_input)

    print("提取 embedding")
    user_embs = trainer.get_user_embedding(test_model_input)

    all_item_ids = item_profile['click_article_id'].values
    all_item_categories = item_profile['category_id'].values
    all_item_words = item_profile['words_count'].values

    item_embs = trainer.get_item_embedding(
        all_item_ids,
        item_category_ids=all_item_categories,
        item_words_ids=all_item_words
    )

    user_embs = user_embs / np.linalg.norm(user_embs, axis=1, keepdims=True)
    item_embs = item_embs / np.linalg.norm(item_embs, axis=1, keepdims=True)

    raw_user_id_emb_dict = {
        user_index_2_rawid[k]: v
        for k, v in zip(test_model_input['user_id'], user_embs)
    }

    raw_item_id_emb_dict = {
        item_index_2_rawid[k]: v
        for k, v in zip(item_profile['click_article_id'], item_embs)
    }

    pickle.dump(raw_user_id_emb_dict, open(save_path + 'user_youtube_emb.pkl', 'wb'))
    pickle.dump(raw_item_id_emb_dict, open(save_path + 'item_youtube_emb.pkl', 'wb'))

    print("向量召回")
    sim, idx = trainer.recall(user_embs, item_embs, topk)

    user_recall_items_dict = collections.defaultdict(dict)

    for target_idx, sim_value_list, rele_idx_list in tqdm(zip(test_model_input['user_id'], sim, idx)):
        target_raw_id = user_index_2_rawid[target_idx]

        for rele_idx, sim_value in zip(rele_idx_list[1:], sim_value_list[1:]):
            rele_raw_id = item_index_2_rawid[item_profile['click_article_id'].iloc[rele_idx]]
            user_recall_items_dict[target_raw_id][rele_raw_id] = \
                user_recall_items_dict.get(target_raw_id, {}).get(rele_raw_id, 0) + sim_value

    user_recall_items_dict = {
        k: sorted(v.items(), key=lambda x: x[1], reverse=True)
        for k, v in user_recall_items_dict.items()
    }

    print("保存结果")
    pickle.dump(user_recall_items_dict, open(save_path + 'youtube_u2i_dict.pkl', 'wb'))

    return user_recall_items_dict


In [30]:
# 由于这里需要做召回评估，所以讲训练集中的最后一次点击都提取了出来
if not metric_recall:
    user_multi_recall_dict['youtubednn_recall'] = youtubednn_u2i_dict(all_click_df, topk=20)
else:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
    user_multi_recall_dict['youtubednn_recall'] = youtubednn_u2i_dict(trn_hist_click_df, topk=20)
    # 召回效果评估
    metrics_recall(user_multi_recall_dict['youtubednn_recall'], trn_last_click_df, topk=20)

构造数据集


100%|██████████| 250000/250000 [00:12<00:00, 19806.16it/s]


开始训练


Epoch 1/10: 100%|██████████| 2104/2104 [00:15<00:00, 132.76it/s]


[Trainer] epoch 1 loss: 11226.4401


...

[Trainer] epoch 9 loss: 8855.7547


Epoch 10/10: 100%|██████████| 2104/2104 [00:17<00:00, 123.26it/s]


[Trainer] epoch 10 loss: 8740.5536
提取 embedding
向量召回


250000it [00:57, 4355.40it/s]


保存结果
 topk:  10  :  hit_num:  17790 hit_rate:  0.07116 user_num :  250000
 topk:  20  :  hit_num:  25563 hit_rate:  0.10225 user_num :  250000


In [20]:
from models.CF import user_based_recommend

In [21]:
# # 读取YoutubeDNN过程中产生的user embedding, 然后使用faiss计算用户之间的相似度
# # 这里需要注意，这里得到的user embedding其实并不是很好，因为YoutubeDNN中使用的是用户点击序列来训练的user embedding,
# # 如果序列普遍都比较短的话，其实效果并不是很好
# user_emb_dict = pickle.load(open(save_path + 'user_youtube_emb.pkl', 'rb'))
# u2u_sim = u2u_embdding_sim(all_click_df, user_emb_dict, save_path, topk=10)

In [22]:
# emb_i2i_sim = pickle.load(open(save_path + 'emb_i2i_sim.pkl', 'rb'))

In [ ]:
# # 使用召回评估函数验证当前召回方式的效果
# if metric_recall:
#     trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
# else:
#     trn_hist_click_df = all_click_df

# user_recall_items_dict = collections.defaultdict(dict)
# user_item_time_dict = get_user_item_time(trn_hist_click_df)
# u2u_sim = pickle.load(open(save_path + 'youtube_u2u_sim.pkl', 'rb'))

# sim_user_topk = 20
# recall_item_num = 10

# item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)
# for user in tqdm(trn_hist_click_df['user_id'].unique()):
#     user_recall_items_dict[user] = user_based_recommend(user, user_item_time_dict, u2u_sim, sim_user_topk, \
#                                                         recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)
    
# user_multi_recall_dict['youtubednn_usercf_recall'] = user_recall_items_dict
# pickle.dump(user_multi_recall_dict['youtubednn_usercf_recall'], open(save_path + 'youtubednn_usercf_recall.pkl', 'wb'))

# if metric_recall:
#     # 召回效果评估
#     metrics_recall(user_multi_recall_dict['youtubednn_usercf_recall'], trn_last_click_df, topk=recall_item_num)